# 📊 N-gram Word Predictor — Train on OpenWebText

Trains a trigram language model with linear interpolation on **1% of OpenWebText (~100M tokens)**.

### Features
- **No GPU needed** — runs on CPU in a few minutes
- **Google Drive persistence** — model is saved to Drive
- **Test predictions** — compare with transformer model

No special runtime needed — CPU is fine.

In [ ]:
# ============================================================
# 1. SETUP & MOUNT DRIVE
# ============================================================
!pip install -q datasets

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/Wordprediktor'
DRIVE_MODEL_DIR = os.path.join(DRIVE_BASE, 'models', 'ngram-openwebtext')
MODEL_FILE = os.path.join(DRIVE_MODEL_DIR, 'model.json')
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print(f'Model will be saved to: {MODEL_FILE}')

In [ ]:
# ============================================================
# 2. N-GRAM MODEL CLASS
# ============================================================
import json
import re
import math
import time
from collections import Counter
from typing import List, Tuple

_WORD_RE = re.compile(r"[a-zA-Z]+(?:['-][a-zA-Z]+)*")

def tokenize(text: str) -> List[str]:
    """Extract words, keeping internal apostrophes and hyphens."""
    return [m.group().lower() for m in _WORD_RE.finditer(text)]


class NgramModel:
    """Trigram language model with linear interpolation smoothing."""

    def __init__(self, lambdas=(0.1, 0.3, 0.6), min_count=2):
        assert abs(sum(lambdas) - 1.0) < 1e-6
        self.lambdas = lambdas
        self.min_count = min_count
        self.unigram_counts = Counter()
        self.bigram_counts = Counter()
        self.trigram_counts = Counter()
        self.vocab = set()
        self.total_tokens = 0
        self._bigram_context = Counter()
        self._trigram_context = Counter()

    def train(self, texts):
        raw_counts = Counter()
        for text in texts:
            words = tokenize(text)
            raw_counts.update(words)
            for i, w in enumerate(words):
                self.unigram_counts[w] += 1
                self.total_tokens += 1
                if i >= 1:
                    bg = (words[i-1], w)
                    self.bigram_counts[bg] += 1
                    self._bigram_context[words[i-1]] += 1
                if i >= 2:
                    tg = (words[i-2], words[i-1], w)
                    self.trigram_counts[tg] += 1
                    self._trigram_context[(words[i-2], words[i-1])] += 1
        self.vocab = {w for w, c in raw_counts.items() if c >= self.min_count}
        print(f'  Vocabulary size : {len(self.vocab):,}')
        print(f'  Total tokens    : {self.total_tokens:,}')
        print(f'  Unique bigrams  : {len(self.bigram_counts):,}')
        print(f'  Unique trigrams : {len(self.trigram_counts):,}')

    def log_prob(self, word, context=()):
        p = self._interpolated_prob(word, context)
        return math.log10(p) if p > 0 else -99.0

    def _interpolated_prob(self, word, context):
        l1, l2, l3 = self.lambdas
        p_uni = self.unigram_counts.get(word, 0) / max(self.total_tokens, 1)
        p_bi = 0.0
        if len(context) >= 1:
            w1 = context[-1]
            bg_c = self.bigram_counts.get((w1, word), 0)
            ctx_c = self._bigram_context.get(w1, 0)
            if ctx_c > 0:
                p_bi = bg_c / ctx_c
        p_tri = 0.0
        if len(context) >= 2:
            w1, w2 = context[-2], context[-1]
            tg_c = self.trigram_counts.get((w1, w2, word), 0)
            ctx_c = self._trigram_context.get((w1, w2), 0)
            if ctx_c > 0:
                p_tri = tg_c / ctx_c
        return l1 * p_uni + l2 * p_bi + l3 * p_tri

    def save(self, path):
        os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
        data = {
            'lambdas': list(self.lambdas),
            'min_count': self.min_count,
            'total_tokens': self.total_tokens,
            'vocab': sorted(self.vocab),
            'unigrams': dict(self.unigram_counts),
            'bigrams': {f'{k[0]}|{k[1]}': v for k, v in self.bigram_counts.items()},
            'trigrams': {f'{k[0]}|{k[1]}|{k[2]}': v for k, v in self.trigram_counts.items()},
            'bigram_ctx': dict(self._bigram_context),
            'trigram_ctx': {f'{k[0]}|{k[1]}': v for k, v in self._trigram_context.items()},
        }
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f)
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f'  Model saved to {path} ({size_mb:.1f} MB)')

    @classmethod
    def load(cls, path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        model = cls(lambdas=tuple(data['lambdas']), min_count=data['min_count'])
        model.total_tokens = data['total_tokens']
        model.vocab = set(data['vocab'])
        model.unigram_counts = Counter(data['unigrams'])
        model.bigram_counts = Counter({tuple(k.split('|')): v for k, v in data['bigrams'].items()})
        model.trigram_counts = Counter({tuple(k.split('|')): v for k, v in data['trigrams'].items()})
        model._bigram_context = Counter(data['bigram_ctx'])
        model._trigram_context = Counter({tuple(k.split('|')): v for k, v in data['trigram_ctx'].items()})
        print(f'  Loaded n-gram model: {len(model.vocab):,} words, {model.total_tokens:,} tokens')
        return model

print('NgramModel defined ✓')

In [ ]:
# ============================================================
# 3. LOAD OPENWEBTEXT (1% ≈ 100M tokens)
# ============================================================
from datasets import load_dataset

DATA_PCT = 1  # 1% of OpenWebText ≈ 80K docs ≈ 100M tokens

print(f'Loading OpenWebText ({DATA_PCT}% subset) ...')
ds = load_dataset('Skylion007/openwebtext', split=f'train[:{DATA_PCT}%]')
texts = [row['text'] for row in ds if row['text'].strip()]
print(f'{len(texts):,} documents loaded')

In [ ]:
# ============================================================
# 4. TRAIN & SAVE
# ============================================================
print('Training n-gram model ...')
t0 = time.time()
model = NgramModel(min_count=2)
model.train(texts)
elapsed = time.time() - t0
print(f'  Training took {elapsed:.1f}s')

print('\nSaving model to Google Drive ...')
model.save(MODEL_FILE)

print('\n--- Top 20 unigrams ---')
for w, c in model.unigram_counts.most_common(20):
    print(f'  {w:15s} {c:>10,}')

In [ ]:
# ============================================================
# 5. N-GRAM PREDICTOR
# ============================================================
class NgramPredictor:
    """Word predictor backed by n-gram model. Same API as TransformerPredictor."""

    def __init__(self, model_path):
        print(f'[NgramPredictor] Loading {model_path}')
        self.model = NgramModel.load(model_path)
        self._vocab_by_freq = sorted(
            self.model.vocab,
            key=lambda w: self.model.unigram_counts.get(w, 0),
            reverse=True,
        )

    def predict(self, context, prefix='', top_k=5):
        context_words = tokenize(context)
        ctx = tuple(context_words[-2:]) if len(context_words) >= 2 else tuple(context_words)
        prefix_lower = prefix.lower()
        candidates = []
        for word in self._vocab_by_freq:
            if prefix_lower and not word.startswith(prefix_lower):
                continue
            score = self.model.log_prob(word, ctx)
            candidates.append((score, word))
        candidates.sort(key=lambda x: x[0], reverse=True)
        seen = set()
        results = []
        for score, word in candidates:
            if word not in seen:
                seen.add(word)
                results.append(word)
            if len(results) >= top_k:
                break
        return results

    def predict_from_text(self, typed_text, top_k=5):
        if not typed_text:
            return []
        if typed_text.endswith(' '):
            return self.predict(typed_text, '', top_k=top_k)
        parts = typed_text.rsplit(' ', 1)
        if len(parts) == 1:
            return self.predict('', parts[0], top_k=top_k)
        return self.predict(parts[0] + ' ', parts[1], top_k=top_k)

print('NgramPredictor defined ✓')

In [ ]:
# ============================================================
# 6. TEST PREDICTIONS
# ============================================================
predictor = NgramPredictor(MODEL_FILE)

tests = [
    'The quick brown ',
    'The quick br',
    'Machine learning is a ',
    'I went to the ',
    'pyt',
    'Artificial intelli',
    'The president of the ',
    'In the year ',
    'I want to ',
    'She said that ',
]

for t in tests:
    s = predictor.predict_from_text(t, top_k=5)
    print(f"  '{t}'  ->  {s}")

In [ ]:
# ============================================================
# 7. INTERACTIVE TEST
# ============================================================
while True:
    text = input('\nType (or quit): ')
    if text.lower() == 'quit':
        break
    print(f'  -> {predictor.predict_from_text(text, top_k=5)}')